# Week 22: Computer Science

Let's explore the world of "pure" computer science through a classic graph theory problem.

Given some nodes (A, B, C, D, E, F) and connections between them (also known as "edges"), what is the shortest path from A to F?

In [ ]:
# Let's first define our graph

graph = {
    "A": [("B", 4), ("C", 2)],
    "B": [("D", 5), ("E", 1)],
    "C": [("B", 1), ("E", 7)],
    "D": [("F", 3)],
    "E": [("F", 1)],
    "F": []
}

### Brute Force version (with recursion)

In [ ]:
def shortest_path(graph, current, goal, visited=set(), depth=0):

    # Firstly, let's take care of the case where we have arrived at the goal.
    if current == goal:
        return ([goal], 0)
    
    # If we aren't at the goal, update the list of visited nodes with the one we have currently arrived at.
    visited.add(current)

    # We now iterate through all subsequent paths and their corresponding costs to find the best one
    best_cost = float("inf")
    best_path = None

    # Look at all the neighbours (and the corresponding weights)
    for neighbour, weight in graph[current]:
        # But only try them if we haven't yet visited them
        if neighbour not in visited:
            # Recursively get the shortest path from the neighbour to the goal:
            path, cost = shortest_path(graph, neighbour, goal, visited=visited.copy(), depth = depth+1)
            # Add this cost to the weight of getting to the current neighbour (the one we are iterating on)
            total_cost = weight + cost
            # If we've found a better path, update the best_cost and best_path
            if total_cost < best_cost:
                best_cost = total_cost
                best_path = [current] + path
                
    return best_path, best_cost

### Dynamic Programming (Floyd–Warshall Algorithm)

In [ ]:
# If we use dynamic programming we will have a distance table, so it's useful to have the nodes be digits so we can use them as indices.
# This graph is identical, but it is with nodes (0,1,2,3,4,5) as opposed to (A, B, C, D, E, F)

graph = {
    0: [(1, 4), (2, 2)],
    1: [(3, 5), (4, 1)],
    2: [(1, 1), (4, 7)],
    3: [(5, 3)],
    4: [(5, 1)],
    5: []
}

In [ ]:
import numpy as np

def shortest_path_dynamic(graph, start, end):

    N = len(graph)

    # This table represents the best known length between nodes.
    # As we are just starting, they are all infinities except the diagonal, which has zeros (the cost to go from node 3 to node 3 is zero, etc.)
    M = np.ones((N, N))*float("inf")
    np.fill_diagonal(M, 0)

    # This is an optional table which stores the "best" node to go to next. It can be used for path reconstruction
    next_node = np.ones((N,N))*float("inf")
    for i in range(N):
        next_node[i,i] = i

    # Update the table with the weights between nodes as listed in our graph
    for i in range(N):
        for neighbour, weight in graph[i]:
            M[i,neighbour] = weight
            next_node[i, neighbour] = i  # Optional, for path reconstruction

    # See if the cost would improve if we consider ever node a possible intermediate for every other node
    for intermediate in range(N):
        for i in range(N):
            for j in range(N):
                if M[i,j] > M[i,intermediate] + M[intermediate,j]:  # If we've found a shorter path via an intermediate...
                    M[i,j] = M[i,intermediate] + M[intermediate,j]  # ...then update the distance table.
                    next_node[i,j] = next_node[intermediate,j]   # Optional, for path reconstruction

    # Go through the path table to reconstruct the optimal path:
    path = [end]
    while start != end:
        end = int(next_node[start][end])
        path = [end] + path

    return M, path

M, path = shortest_path_dynamic(graph, 0, 5)

### Djikstra's Algorithm
A very widely used Greedy algorithm (it always chooses what seems best in the moment, without thinking ahead)

In [53]:
# This implements a "heap queue". It's like a list, but with a special property that we can "pop" out the smallest value
import heapq

def dijkstra(graph, start, goal):

    # Shortest known distance to each node
    distances = {node: float("inf") for node in graph}
    distances[start] = 0

    # Track best path
    previous = {}

    # Priority queue: (distance, node)
    queue = [(0, start)]

    # A list evaluates to true as long as it's not empty
    while queue:
        current_distance, current_node = heapq.heappop(queue)

        # Stop once goal reached
        if current_node == goal:
            break

        # Explore neighbors
        for neighbour, weight in graph[current_node]:
            distance = current_distance + weight

            # See if we have found a shorter distance to any nodes
            if distance < distances[neighbour]:
                distances[neighbour] = distance
                previous[neighbour] = current_node
                heapq.heappush(queue, (distance, neighbour))

    # Reconstruct path
    path = []
    node = goal

    while node != start:
        path.append(node)
        node = previous[node]

    path.append(start)
    path.reverse()

    return distances[goal], path


cost, path = dijkstra(graph, "A", "F")

print("Cost:", cost)
print("Path:", path)

Cost: 5
Path: ['A', 'C', 'B', 'E', 'F']


# Complexity Classes

### $O(1)$ - Constant Time

Simple singular operations

In [24]:
arr = [10, 20, 30, 40]

print(arr[2])

30


### $O(n)$ - Linear Time

Printing every element in an array

In [26]:
arr = [1, 2, 3, 4, 5]

for x in arr:
    print(x)

1
2
3
4
5


### $O(n^{2})$ - Polynomial Time

Printing every combination of 2 numbers from an array of length $n$

In [25]:
arr = [1, 2, 3]

for x in arr:
    for y in arr:
        print(x, y)

1 1
1 2
1 3
2 1
2 2
2 3
3 1
3 2
3 3


### $O(\log{n})$ - Logarithmic Time

Searching for an element in a sorted list by repeatedly cutting the list in half.

$O$ notation is for the "worst case". That is, it's always possible we get lucky and find the target right away, but we may have to keep slicing the list until we have it down to length 1. ie, for a list of length 16 it would be: 16, 8, 4, 2, 1 = 4 operations, which is equal to $\log_2{(16)}$

In [31]:
arr = [1, 3, 5, 7, 9, 11]
target = 7

low = 0
high = len(arr) - 1

while True:
    mid = (low + high) // 2

    if arr[mid] == target:
        print("Found")
        break

    elif arr[mid] < target:
        low = mid + 1

    else:
        high = mid - 1

Found


### $O(n\log{n})$

This is another common complexity class we find. One example is the standard python list sorting algorithm (Tim Sort)

In [32]:
arr = [5, 2, 8, 1, 9]

arr.sort()

print(arr)

[1, 2, 5, 8, 9]


### $O(2^n)$ - Exponential Time

Example: Trying to crack an $n$-digit binary password

In [30]:
def generate(n, s=""):
    if len(s) == n:
        print(s)
        return

    generate(n, s + "0")
    generate(n, s + "1")

generate(3)

000
001
010
011
100
101
110
111


### Regarding our above path optimization algorithms:

| Algorithm | Complexity |
| --- | --- |
| Recursive Brute Force | $O(b^n)$ |
| Floyd–Warshall | $O(n^3)$ |
| Djikstra | $O(n^2)$ |

Notes:
* n = number of nodes
* b = branching factor (average number of neighbours)
* Djikstra's algorithm can be further improved. For example with the priority heap it becomes $O((n + E)\log{n})$, where E is the # of edges

# The Ultimate Question

**P** refers to the "Polynomial Time" class, and is used to denote the group of all problems that can be solved in polynomial time.

**NP** refers to the "Nondeterministic Polynomial Time" class, and is used to denote the group of all problems for which a proposed solution can be quickly verified.

Example:

Given a sudoku, it is difficult to find a solution.

But if you are given a completed sudoku, it is easy to verify that it is indeed a correct solution.

One unsolved question in Computer Science (and certainly the "Holy Grail" of the domain) is whether:

$$\bf{P} = \bf{NP}$$

If $\bf{P} = \bf{NP}$, for every problem where it is possible to verify a solution in polynomial time, then it must also be possible to find a solution in polynomial time.

*If P=NP, then the world would be a profoundly different place than we usually assume it to be. There would be no special value in “creative leaps,” no fundamental gap between solving a problem and recognizing the solution once it’s found. Everyone who could appreciate a symphony would be Mozart; everyone who could follow a step-by-step argument would be Gauss; everyone who could recognize a good investment strategy would be Warren Buffett.*

-- Scott Aaronson